# SH17 YOLO reproduction and model sweep

This notebook is built for Google Colab. It prepares the official SH17 split, validates the published YOLOv9-e checkpoint, then trains/evaluates a sweep that includes YOLOv9-e, YOLO11x, YOLO26l, and YOLO26x.

Target sanity check for the published YOLOv9-e SH17 checkpoint: about 70.9 mAP50 and 48.7 mAP50-95 on the official held-out split.

## References used

- SH17 official repository: https://github.com/ahmadmughees/SH17dataset
- SH17 paper notes: 8,099 images, 75,994 instances, 80/20 train/test split, 200 epochs, image size 640, YOLOv9-e at 70.9 mAP50 / 48.7 mAP50-95.
- Official SH17 trained YOLO weights release: https://github.com/ahmadmughees/SH17dataset/releases/tag/v1
- Ultralytics model names used here: yolov9e.pt, yolo11x.pt, yolo26l.pt, yolo26x.pt.

## 1. Install dependencies

Run this first after selecting a GPU runtime in Colab.

In [ ]:
import os
import platform
import subprocess
import sys


def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], check=True)


pip_install("ultralytics", "kaggle", "pyyaml", "pandas")

print("Python:", sys.version)
print("Platform:", platform.platform())

## 2. Experiment configuration

Defaults below follow the SH17 paper where possible: 200 epochs, image size 640, default Ultralytics augmentation/hyperparameters. Use batch = -1 for Colab auto-batch. If you have an A100/L4 and want to be closer to the paper, try batch = 32 for the large models.

In [ ]:
from pathlib import Path
import gc
import json
import random
import shutil
import time
import urllib.request

import pandas as pd
import torch
import yaml
from ultralytics import YOLO

KAGGLE_DATASET_SLUG = "mugheesahmad/sh17-dataset-for-ppe-detection"
DATASET_ROOT = None  # Set to an existing SH17 root path if you already downloaded the dataset.

USE_GOOGLE_DRIVE = True
WORK_DIR = Path("/content/sh17_work")
PROJECT_DIR = WORK_DIR / "runs"

RUN_OFFICIAL_YOLOV9E_SANITY_VAL = True
RUN_TRAINING_SWEEP = True
STOP_ON_ERROR = False

EPOCHS = 200
IMGSZ = 640
BATCH = -1  # -1 lets Ultralytics pick a safe batch size for the current Colab GPU.
WORKERS = 8
SEED = 0
DETERMINISTIC = True
PATIENCE = 50

DEVICE = 0 if torch.cuda.is_available() else "cpu"
RUN_ID = time.strftime("%Y%m%d-%H%M%S")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        PROJECT_DIR = Path("/content/drive/MyDrive/sh17_runs")
    except Exception as exc:
        print("Drive mount skipped:", repr(exc))
        PROJECT_DIR = WORK_DIR / "runs"

WORK_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_SPECS = [
    {"name": "yolov9e", "weights": "yolov9e.pt", "enabled": True},
    # {"name": "yolo11x", "weights": "yolo11x.pt", "enabled": True},
    # {"name": "yolo26l", "weights": "yolo26l.pt", "enabled": True},
    # {"name": "yolo26x", "weights": "yolo26x.pt", "enabled": True},
]

RESULTS_CSV = PROJECT_DIR / "sh17_results.csv"
RESULTS_JSONL = PROJECT_DIR / "sh17_results.jsonl"

print("Ultralytics device:", DEVICE)
print("Project dir:", PROJECT_DIR)
print("Run id:", RUN_ID)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Download SH17 from Kaggle

If Colab asks for kaggle.json, upload the Kaggle API token from your account settings. If the dataset is already present, set DATASET_ROOT in the config cell and rerun from here.

In [ ]:
def ensure_kaggle_credentials():
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if kaggle_json.exists():
        kaggle_json.chmod(0o600)
        return

    if not IN_COLAB:
        raise FileNotFoundError(
            "Missing ~/.kaggle/kaggle.json. Put your Kaggle API token there or run this notebook in Colab."
        )

    from google.colab import files
    print("Upload kaggle.json now.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("kaggle.json was not uploaded.")
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    shutil.move("kaggle.json", kaggle_json)
    kaggle_json.chmod(0o600)


def find_dataset_root(base_dir: Path) -> Path:
    candidates = []
    for path in [base_dir, *base_dir.rglob("*")]:
        if not path.is_dir():
            continue
        has_split = any((path / name).exists() for name in ("train_files.txt", "val_files.txt", "test_files.txt"))
        has_images = any(child.is_dir() and child.name.lower() == "images" for child in path.iterdir())
        if has_split or has_images:
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f"Could not find an SH17-like dataset root under {base_dir}")
    return sorted(candidates, key=lambda p: len(str(p)))[0]


def download_sh17_dataset(dataset_root=None):
    if dataset_root:
        root = Path(dataset_root).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(root)
        return root

    ensure_kaggle_credentials()
    download_dir = Path("/content/datasets/sh17")
    download_dir.mkdir(parents=True, exist_ok=True)

    if not any(download_dir.rglob("train_files.txt")) and not any(download_dir.rglob("val_files.txt")):
        from kaggle.api.kaggle_api_extended import KaggleApi

        print("Downloading:", KAGGLE_DATASET_SLUG)
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(KAGGLE_DATASET_SLUG, path=str(download_dir), unzip=True)
    else:
        print("Dataset appears to be already downloaded:", download_dir)

    return find_dataset_root(download_dir)


DATASET_ROOT = download_sh17_dataset(DATASET_ROOT)
print("DATASET_ROOT =", DATASET_ROOT)

## 4. Prepare the official split and YOLO data YAML

This cell uses the official train_files.txt and val_files.txt/test_files.txt that come with SH17. It writes absolute-path split files so Colab path changes do not silently corrupt the experiment.

In [ ]:
SH17_NAMES = {
    0: "person",
    1: "ear",
    2: "ear-mufs",
    3: "face",
    4: "face-guard",
    5: "face-mask",
    6: "foot",
    7: "tool",
    8: "glasses",
    9: "gloves",
    10: "helmet",
    11: "hands",
    12: "head",
    13: "medical-suit",
    14: "shoes",
    15: "safety-suit",
    16: "safety-vest",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_SPLIT = {
    "train_images": 6479,
    "val_images": 1620,
    "val_instances": 15358,
    "total_images": 8099,
    "total_instances": 75994,
}
STRICT_OFFICIAL_SPLIT = True


def count_images(path: Path) -> int:
    return sum(1 for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTS)


def count_txt(path: Path) -> int:
    return sum(1 for p in path.rglob("*.txt"))


def find_largest_named_dir(root: Path, name: str, counter) -> Path:
    hits = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() == name]
    if not hits:
        raise FileNotFoundError(f"Could not find directory named {name} under {root}")
    return max(hits, key=counter)


def find_split_file(root: Path, names) -> Path:
    hits = []
    for name in names:
        hits.extend(root.rglob(name))
    if not hits:
        raise FileNotFoundError(f"Could not find any split file among {names} under {root}")
    return sorted(hits, key=lambda p: len(str(p)))[0]


images_dir = find_largest_named_dir(DATASET_ROOT, "images", count_images)
labels_dir = find_largest_named_dir(DATASET_ROOT, "labels", count_txt)
train_split_file = find_split_file(DATASET_ROOT, ["train_files.txt"])
val_split_file = find_split_file(DATASET_ROOT, ["val_files.txt", "test_files.txt"])

print("images_dir:", images_dir)
print("labels_dir:", labels_dir)
print("train_split_file:", train_split_file)
print("val_split_file:", val_split_file)

all_images = [p.resolve() for p in images_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
by_name = {}
by_stem = {}
for img in all_images:
    by_name.setdefault(img.name, img)
    by_stem.setdefault(img.stem, img)


def read_split_entries(path: Path):
    return [line.strip().replace("\\", "/") for line in path.read_text().splitlines() if line.strip()]


def resolve_split_entries(split_path: Path):
    resolved = []
    missing = []
    for entry in read_split_entries(split_path):
        entry_path = Path(entry)
        candidates = []
        if entry_path.is_absolute():
            candidates.append(entry_path)
        candidates.extend([
            DATASET_ROOT / entry,
            images_dir / entry,
            images_dir / entry_path.name,
        ])
        if entry_path.name:
            candidates.append(by_name.get(entry_path.name))
        if entry_path.stem:
            candidates.append(by_stem.get(entry_path.stem))

        found = next((p.resolve() for p in candidates if p is not None and Path(p).exists()), None)
        if found is None:
            missing.append(entry)
        else:
            resolved.append(found)

    if missing:
        preview = missing[:10]
        raise FileNotFoundError(f"Could not resolve {len(missing)} entries from {split_path}: {preview}")
    return resolved


train_images = resolve_split_entries(train_split_file)
val_images = resolve_split_entries(val_split_file)


def label_for_image(image_path: Path) -> Path:
    flat_label = labels_dir / f"{image_path.stem}.txt"
    try:
        rel = image_path.resolve().relative_to(images_dir.resolve())
        mirrored_label = (labels_dir / rel).with_suffix(".txt")
        return mirrored_label if mirrored_label.exists() else flat_label
    except ValueError:
        return flat_label


def count_instances(image_paths):
    missing_labels = []
    total = 0
    for img in image_paths:
        label_path = label_for_image(img)
        if not label_path.exists():
            missing_labels.append(str(label_path))
            continue
        with label_path.open("r", encoding="utf-8") as f:
            total += sum(1 for line in f if line.strip())
    return total, missing_labels


train_instances, missing_train_labels = count_instances(train_images)
val_instances, missing_val_labels = count_instances(val_images)

summary = {
    "train_images": len(train_images),
    "val_images": len(val_images),
    "total_images": len(set(train_images + val_images)),
    "train_instances": train_instances,
    "val_instances": val_instances,
    "total_instances": train_instances + val_instances,
    "missing_train_labels": len(missing_train_labels),
    "missing_val_labels": len(missing_val_labels),
}
print(json.dumps(summary, indent=2))

checks = {
    "train_images": summary["train_images"] == EXPECTED_SPLIT["train_images"],
    "val_images": summary["val_images"] == EXPECTED_SPLIT["val_images"],
    "val_instances": summary["val_instances"] == EXPECTED_SPLIT["val_instances"],
    "total_images": summary["total_images"] == EXPECTED_SPLIT["total_images"],
    "total_instances": summary["total_instances"] == EXPECTED_SPLIT["total_instances"],
    "missing_labels": summary["missing_train_labels"] == 0 and summary["missing_val_labels"] == 0,
}
print("Official split checks:", json.dumps(checks, indent=2))

if STRICT_OFFICIAL_SPLIT and not all(checks.values()):
    raise AssertionError("Official SH17 split validation failed. Do not train until the dataset/split files are fixed.")

SPLIT_DIR = WORK_DIR / "official_split"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_ABS = SPLIT_DIR / "train_files_abs.txt"
VAL_ABS = SPLIT_DIR / "val_files_abs.txt"
DATA_YAML = SPLIT_DIR / "sh17_official.yaml"

TRAIN_ABS.write_text("\n".join(str(p) for p in train_images) + "\n", encoding="utf-8")
VAL_ABS.write_text("\n".join(str(p) for p in val_images) + "\n", encoding="utf-8")

DATA_YAML.write_text(
    yaml.safe_dump(
        {
            "path": "/",
            "train": str(TRAIN_ABS),
            "val": str(VAL_ABS),
            "test": str(VAL_ABS),
            "names": SH17_NAMES,
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)

print("DATA_YAML =", DATA_YAML)
print(DATA_YAML.read_text())

## 5. Metric helpers

All validations and training runs append to sh17_results.csv and sh17_results.jsonl in PROJECT_DIR.

In [ ]:
def safe_float(value):
    try:
        return float(value)
    except Exception:
        return None


def metric_fitness(metrics):
    value = getattr(metrics, "fitness", None)
    if callable(value):
        value = value()
    return safe_float(value)


def metrics_to_record(model_name, metrics, weights_path=None, status="ok", extra=None):
    box = getattr(metrics, "box", None)
    record = {
        "run_id": RUN_ID,
        "model": model_name,
        "status": status,
        "weights": str(weights_path) if weights_path else "",
        "map50": safe_float(getattr(box, "map50", None)),
        "map50_95": safe_float(getattr(box, "map", None)),
        "precision": safe_float(getattr(box, "mp", None)),
        "recall": safe_float(getattr(box, "mr", None)),
        "fitness": metric_fitness(metrics),
        "imgsz": IMGSZ,
        "batch": BATCH,
        "epochs": EPOCHS,
        "device": str(DEVICE),
        "ultralytics_version": __import__("ultralytics").__version__,
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    if extra:
        record.update(extra)
    return record


def append_record(record):
    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame([record])
    df.to_csv(RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False)
    with RESULTS_JSONL.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=True) + "\n")
    print(json.dumps(record, indent=2, ensure_ascii=True))
    print("Updated:", RESULTS_CSV)


def show_results_tail(n=20):
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        display(df.tail(n))
    else:
        print("No results CSV yet:", RESULTS_CSV)

## 6. Sanity-check the published YOLOv9-e SH17 checkpoint

This is the fastest way to confirm the official split and metric pipeline. The expected values are approximately mAP50 = 0.709 and mAP50-95 = 0.487.

In [ ]:
OFFICIAL_YOLOV9E_URL = "https://github.com/ahmadmughees/SH17dataset/releases/download/v1/yolo9e.pt"
OFFICIAL_YOLOV9E_PATH = WORK_DIR / "weights" / "sh17_official_yolo9e.pt"


def download_file(url, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        print("Already exists:", path)
        return path
    print("Downloading:", url)
    urllib.request.urlretrieve(url, path)
    print("Saved:", path, "size:", path.stat().st_size)
    return path


if RUN_OFFICIAL_YOLOV9E_SANITY_VAL:
    official_weights = download_file(OFFICIAL_YOLOV9E_URL, OFFICIAL_YOLOV9E_PATH)
    official_model = YOLO(str(official_weights))
    official_metrics = official_model.val(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        batch=1,
        device=DEVICE,
        split="val",
        project=str(PROJECT_DIR / "val"),
        name="official_yolo9e_sh17_val",
        exist_ok=True,
        plots=True,
    )
    record = metrics_to_record(
        "official_sh17_yolo9e_val",
        official_metrics,
        weights_path=official_weights,
        extra={"target_map50": 0.709, "target_map50_95": 0.487, "phase": "published_weight_val"},
    )
    append_record(record)
else:
    print("RUN_OFFICIAL_YOLOV9E_SANITY_VAL is False")

show_results_tail()

## 7. Train and evaluate the requested model sweep

This trains from the public COCO-pretrained weights for each architecture, then validates the best checkpoint on the official SH17 validation/test split. The four requested models are enabled by default: YOLOv9-e, YOLO11x, YOLO26l, and YOLO26x.

For a quick smoke test, set EPOCHS = 1 and leave only one model enabled in MODEL_SPECS. For reproduction-quality runs, keep EPOCHS = 200.

In [ ]:
def train_and_eval_one(spec):
    model_name = spec["name"]
    weights = spec["weights"]
    epochs = spec.get("epochs", EPOCHS)
    batch = spec.get("batch", BATCH)
    start = time.time()

    print("=" * 80)
    print(f"Training {model_name} from {weights}")
    print("=" * 80)

    model = YOLO(weights)
    model.train(
        data=str(DATA_YAML),
        epochs=epochs,
        imgsz=IMGSZ,
        batch=batch,
        device=DEVICE,
        workers=WORKERS,
        seed=SEED,
        deterministic=DETERMINISTIC,
        patience=PATIENCE,
        pretrained=True,
        val=True,
        plots=True,
        project=str(PROJECT_DIR / "train"),
        name=model_name,
        exist_ok=True,
    )

    save_dir = Path(getattr(getattr(model, "trainer", None), "save_dir", PROJECT_DIR / "train" / model_name))
    best_pt = save_dir / "weights" / "best.pt"
    last_pt = save_dir / "weights" / "last.pt"
    eval_weights = best_pt if best_pt.exists() else last_pt
    if not eval_weights.exists():
        raise FileNotFoundError(f"No checkpoint found for {model_name} in {save_dir}")

    print(f"Validating {model_name} checkpoint: {eval_weights}")
    val_metrics = YOLO(str(eval_weights)).val(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        batch=1,
        device=DEVICE,
        split="val",
        project=str(PROJECT_DIR / "val"),
        name=f"{model_name}_best_val",
        exist_ok=True,
        plots=True,
    )

    elapsed_min = (time.time() - start) / 60.0
    return metrics_to_record(
        model_name,
        val_metrics,
        weights_path=eval_weights,
        extra={
            "phase": "train_then_val",
            "source_weights": weights,
            "train_save_dir": str(save_dir),
            "elapsed_min": round(elapsed_min, 2),
            "epochs": epochs,
            "batch": batch,
        },
    )


if RUN_TRAINING_SWEEP:
    active_specs = [spec for spec in MODEL_SPECS if spec.get("enabled", True)]
    print("Active model specs:", active_specs)
    for spec in active_specs:
        try:
            record = train_and_eval_one(spec)
            append_record(record)
        except Exception as exc:
            error_record = {
                "run_id": RUN_ID,
                "model": spec.get("name", "unknown"),
                "status": "failed",
                "phase": "train_then_val",
                "source_weights": spec.get("weights", ""),
                "error": repr(exc),
                "time": time.strftime("%Y-%m-%d %H:%M:%S"),
            }
            append_record(error_record)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if STOP_ON_ERROR:
                raise
else:
    print("RUN_TRAINING_SWEEP is False")

show_results_tail()

## 8. What to send back

After the Colab run finishes, send back the last rows of sh17_results.csv plus the run folders for any model that looks promising. The key columns are model, status, map50, map50_95, precision, recall, source_weights, and train_save_dir.

In [ ]:
show_results_tail(50)
print("Results CSV:", RESULTS_CSV)
print("Results JSONL:", RESULTS_JSONL)
print("Project directory:", PROJECT_DIR)